## Imports

In [1]:
%load_ext autoreload
%autoreload 2

# Standard imports
import glob

# 3rd party imports
import cv2
import matplotlib.pyplot as plt
import numpy as np 
from pprint import pprint
import SimpleITK as sitk

In [ ]:
# Published implementation, frozen: velazquez_rivera_2025/ at the repository root.
# New work belongs in vessel_utils/ instead; this package reproduces the paper.
# `pip install -e .` there makes the import below work anywhere; this is the fallback.
import pathlib, sys

_root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "pyproject.toml").exists()), None)
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from velazquez_rivera_2025.io import read_tif
from velazquez_rivera_2025.enhance import auto_contrast, gamma_correction
from velazquez_rivera_2025.vessels import detect_vessels, get_brain_mask, process_vessels
from velazquez_rivera_2025.metrics import dice_coefficient, iou, precision, rand_index, recall


## Functions

In [3]:
import itk
import numpy as np
from skimage.morphology import remove_small_objects, binary_closing, disk, remove_small_holes

## Load data

In [ ]:
m1_path = "/media/data/u01/Endothelial_enhancer/Fig3_M14_*.tif"
m1_data = sorted(glob.glob(m1_path))
m1_data

Apply exponential scaling on the image

In [ ]:
IDX = 0
curr_img = read_tif(m1_data[IDX])
print(f"Image shape: {curr_img.shape}")
curr_ch0 = curr_img[0]
curr_ch1 = curr_img[1]
curr_ch2 = curr_img[2]

# Check image stats
print(f"Image shape: {curr_ch0.shape}")
print(f"Image min: {curr_ch0.min()}")
print(f"Image max: {curr_ch0.max()}")
print(f"Image mean: {curr_ch0.mean()}")


# Ch0 settings
gamma_ch0 = 2  # You can adjust this value to control the contrast enhancement
contrast_alpha_ch0 = 0.5  # Try 0.15 You can adjust this value to control the brightness enhancement 0.5 default

# Ch1 settings
gamma_ch1 = 2  # You can adjust this value to control the contrast enhancement
contrast_alpha_ch1 = 1.5  # Try 0.15 You can adjust this value to control the brightness enhancement 0.5 default

# No change
contrast_ch0 = curr_ch0
contrast_ch1 = curr_ch1
contrast_ch2 = curr_ch2

# Compute the original image contrast
#contrast_ch0 = gamma_correction(curr_ch0, gamma=gamma_ch0)
#contrast_ch0 = auto_contrast(contrast_ch0, alpha=contrast_alpha_ch0)
#contrast_ch1 = gamma_correction(curr_ch1, gamma=gamma_ch1)
#contrast_ch1 = auto_contrast(contrast_ch1, alpha=contrast_alpha_ch1)

bg_mask = gamma_correction(curr_ch0, gamma=gamma_ch0)
bg_mask = auto_contrast(bg_mask, alpha=3.0)
bg_mask = get_brain_mask(bg_mask, area_threshold=100000)  # 255 default ch0, 150 for ch1

plt.figure(figsize=(10, 10))
plt.imshow(contrast_ch0, cmap='gray')
plt.contour(bg_mask, colors='red', linewidths=0.5, alpha=0.35)
plt.title(f"Section {IDX} contrast ch0")
plt.axis('off')
plt.show()

plt.figure(figsize=(10, 10))
plt.imshow(contrast_ch1, cmap='gray')
plt.contour(bg_mask, colors='red', linewidths=0.5, alpha=0.35)
plt.title(f"Section {IDX} contrast ch1")
plt.axis('off')
plt.show()

plt.figure(figsize=(10, 10))
plt.imshow(contrast_ch2, cmap='gray')
plt.contour(bg_mask, colors='red', linewidths=0.5, alpha=0.35)
plt.title(f"Section {IDX} contrast ch2")
plt.axis('off')
plt.show()

## Hessian Filter
https://examples.itk.org/src/nonunit/review/segmentbloodvesselswithmultiscalehessianbasedmeasure/documentation

In [ ]:
# Parameters for vessel detection
sigma_minimum = 1.0  # Range of scales in which MultiScaleHessianBasedMeasureImageFilter will search for vessels
sigma_maximum = 10.0  # 10
number_of_sigma_steps = 10  # 10 Number of scales to search for vessels

# Parameters for post-processing
thresh = 230  # Threshold for binarization, 230
min_size = 10  # Minimum size of objects to keep
area_threshold = 2000 # Minimum area of holes to keep
smoothing = 1  # Smoothing factor for closing, 3

xlim = [4000, 5000]  # 8000, 9000 or 9000, 10000
ylim = [4000, 5000]  # 8000, 9000

#############################################################

# Alternative: load image in memory
input_image = contrast_ch0 * bg_mask
input_image = input_image.astype(np.float32)
#input_image *= 255.0

# Print statistics
print("Input image type:", input_image.dtype)
print("Input image min:", input_image.min())
print("Input image max:", input_image.max())

# Run the vessel detection
segmented_vessels_array = detect_vessels(input_image, sigma_minimum, sigma_maximum, number_of_sigma_steps, alpha=0.5, beta=0.5, gamma=5.0)

# Process the thresholded vessels
thresholded_vessels_ch0 = process_vessels(segmented_vessels_array, thresh=thresh, min_size=min_size, area_threshold=area_threshold, smoothing=smoothing)
thresholded_vessels_ch0 = thresholded_vessels_ch0 * bg_mask

# Print statistics
print("Vesselness image statistics:")
print("Shape:", segmented_vessels_array.shape)
print("Min:", segmented_vessels_array.min())
print("Max:", segmented_vessels_array.max())
print("Mean:", segmented_vessels_array.mean())
print("Median:", np.median(segmented_vessels_array))
#print("Std:", segmented_vessels_array.std())

# Plot the raw vesselness image
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(segmented_vessels_array, cmap='gray')
plt.axis('off')
plt.title("Vesselness image")
plt.subplot(1, 2, 2)
plt.imshow(thresholded_vessels_ch0, cmap='gray')
plt.axis('off')
plt.title("Vessel mask")
plt.show()

# Show the results
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(input_image, cmap='gray')
plt.axis('off')
plt.title("Contrast image")
plt.subplot(1, 2, 2)
plt.imshow(input_image, cmap='gray')
plt.contour(thresholded_vessels_ch0, colors='red', linewidths=0.5, alpha=0.45)
plt.axis('off')
plt.title("Vessel mask contours over contrast image")
plt.show()

# Plot the raw vesselness image
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(segmented_vessels_array, cmap='gray')
plt.axis('off')
plt.title("Vesselness image")
plt.xlim(xlim)
plt.ylim(ylim)
plt.subplot(1, 2, 2)
plt.imshow(thresholded_vessels_ch0, cmap='gray')
plt.axis('off')
plt.title("Vessel mask")
plt.xlim(xlim)
plt.ylim(ylim)
plt.show()

# Show the results
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(input_image, cmap='gray')
plt.axis('off')
plt.title("Contrast image")
plt.xlim(xlim)
plt.ylim(ylim)
plt.subplot(1, 2, 2)
plt.imshow(input_image, cmap='gray')
plt.contour(thresholded_vessels_ch0, colors='red', linewidths=0.5, alpha=0.45)
plt.axis('off')
plt.title("Vessel mask contours over contrast image")
plt.xlim(xlim)
plt.ylim(ylim)
plt.show()

# Show the results
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(input_image, cmap='gray')
plt.axis('off')
plt.title("2: Contrast image")
plt.xlim(ylim)
plt.ylim(ylim)
plt.subplot(1, 2, 2)
plt.imshow(input_image, cmap='gray')
plt.contour(thresholded_vessels_ch0, colors='red', linewidths=0.5, alpha=0.45)
plt.axis('off')
plt.title("2: Vessel mask contours over contrast image")
plt.xlim(ylim)
plt.ylim(ylim)
plt.show()

# Show the results
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(input_image, cmap='gray')
plt.axis('off')
plt.title("3: Contrast image")
plt.xlim([2000, 3000])
plt.ylim([2000, 3000])
plt.subplot(1, 2, 2)
plt.imshow(input_image, cmap='gray')
plt.contour(thresholded_vessels_ch0, colors='red', linewidths=0.5, alpha=0.45)
plt.axis('off')
plt.title("3: Vessel mask contours over contrast image")
plt.xlim([2000, 3000])
plt.ylim([2000, 3000])
plt.show()

Repeat for ch1

In [ ]:
# Parameters for vessel detection
sigma_minimum = 1.0  # Range of scales in which MultiScaleHessianBasedMeasureImageFilter will search for vessels
sigma_maximum = 10.0  # 10
number_of_sigma_steps = 10  # 10 Number of scales to search for vessels

# Parameters for post-processing
thresh = 230  # Threshold for binarization, 230
min_size = 10  # Minimum size of objects to keep
area_threshold = 2000 # Minimum area of holes to keep
smoothing = 1  # Smoothing factor for closing, 3

xlim = [4000, 5000]  # 8000, 9000 or 9000, 10000
ylim = [4000, 5000]  # 8000, 9000

#############################################################

# Alternative: load image in memory
input_image = contrast_ch1 * bg_mask
input_image = input_image.astype(np.float32)
#input_image *= 255.0

# Print statistics
print("Input image type:", input_image.dtype)
print("Input image min:", input_image.min())
print("Input image max:", input_image.max())

# Run the vessel detection
segmented_vessels_array = detect_vessels(input_image, sigma_minimum, sigma_maximum, number_of_sigma_steps, alpha=0.5, beta=0.5, gamma=5.0)

# Process the thresholded vessels
thresholded_vessels_ch1 = process_vessels(segmented_vessels_array, thresh=thresh, min_size=min_size, area_threshold=area_threshold, smoothing=smoothing)
thresholded_vessels_ch1 = thresholded_vessels_ch1 * bg_mask

# Print statistics
print("Vesselness image statistics:")
print("Shape:", segmented_vessels_array.shape)
print("Min:", segmented_vessels_array.min())
print("Max:", segmented_vessels_array.max())
print("Mean:", segmented_vessels_array.mean())
print("Median:", np.median(segmented_vessels_array))
#print("Std:", segmented_vessels_array.std())

# Plot the raw vesselness image
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(segmented_vessels_array, cmap='gray')
plt.axis('off')
plt.title("Vesselness image")
plt.subplot(1, 2, 2)
plt.imshow(thresholded_vessels_ch1, cmap='gray')
plt.axis('off')
plt.title("Vessel mask")
plt.show()

# Show the results
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(input_image, cmap='gray')
plt.axis('off')
plt.title("Contrast image")
plt.subplot(1, 2, 2)
plt.imshow(input_image, cmap='gray')
plt.contour(thresholded_vessels_ch1, colors='red', linewidths=0.5, alpha=0.45)
plt.axis('off')
plt.title("Vessel mask contours over contrast image")
plt.show()

# Plot the raw vesselness image
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(segmented_vessels_array, cmap='gray')
plt.axis('off')
plt.title("Vesselness image")
plt.xlim(xlim)
plt.ylim(ylim)
plt.subplot(1, 2, 2)
plt.imshow(thresholded_vessels_ch1, cmap='gray')
plt.axis('off')
plt.title("Vessel mask")
plt.xlim(xlim)
plt.ylim(ylim)
plt.show()

# Show the results
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(input_image, cmap='gray')
plt.axis('off')
plt.title("Contrast image")
plt.xlim(xlim)
plt.ylim(ylim)
plt.subplot(1, 2, 2)
plt.imshow(input_image, cmap='gray')
plt.contour(thresholded_vessels_ch1, colors='red', linewidths=0.5, alpha=0.45)
plt.axis('off')
plt.title("Vessel mask contours over contrast image")
plt.xlim(xlim)
plt.ylim(ylim)
plt.show()

# Show the results
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(input_image, cmap='gray')
plt.axis('off')
plt.title("2: Contrast image")
plt.xlim(ylim)
plt.ylim(ylim)
plt.subplot(1, 2, 2)
plt.imshow(input_image, cmap='gray')
plt.contour(thresholded_vessels_ch1, colors='red', linewidths=0.5, alpha=0.45)
plt.axis('off')
plt.title("2: Vessel mask contours over contrast image")
plt.xlim(ylim)
plt.ylim(ylim)
plt.show()

# Show the results
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(input_image, cmap='gray')
plt.axis('off')
plt.title("3: Contrast image")
plt.xlim([2000, 3000])
plt.ylim([2000, 3000])
plt.subplot(1, 2, 2)
plt.imshow(input_image, cmap='gray')
plt.contour(thresholded_vessels_ch1, colors='red', linewidths=0.5, alpha=0.45)
plt.axis('off')
plt.title("3: Vessel mask contours over contrast image")
plt.xlim([2000, 3000])
plt.ylim([2000, 3000])
plt.show()

In [ ]:
# Parameters for vessel detection
sigma_minimum = 1.0  # Range of scales in which MultiScaleHessianBasedMeasureImageFilter will search for vessels
sigma_maximum = 10.0  # 10
number_of_sigma_steps = 10  # 10 Number of scales to search for vessels

# Parameters for post-processing
thresh = 230  # Threshold for binarization, 230
min_size = 10  # Minimum size of objects to keep
area_threshold = 2000 # Minimum area of holes to keep
smoothing = 1  # Smoothing factor for closing, 3

xlim = [4000, 5000]  # 8000, 9000 or 9000, 10000
ylim = [4000, 5000]  # 8000, 9000

#############################################################

# Alternative: load image in memory
input_image = contrast_ch2 * bg_mask
input_image = input_image.astype(np.float32)
#input_image *= 255.0

# Print statistics
print("Input image type:", input_image.dtype)
print("Input image min:", input_image.min())
print("Input image max:", input_image.max())

# Run the vessel detection
segmented_vessels_array = detect_vessels(input_image, sigma_minimum, sigma_maximum, number_of_sigma_steps, alpha=0.5, beta=0.5, gamma=5.0)

# Process the thresholded vessels
thresholded_vessels_ch2 = process_vessels(segmented_vessels_array, thresh=thresh, min_size=min_size, area_threshold=area_threshold, smoothing=smoothing)
thresholded_vessels_ch2 = thresholded_vessels_ch2 * bg_mask

# Print statistics
print("Vesselness image statistics:")
print("Shape:", segmented_vessels_array.shape)
print("Min:", segmented_vessels_array.min())
print("Max:", segmented_vessels_array.max())
print("Mean:", segmented_vessels_array.mean())
print("Median:", np.median(segmented_vessels_array))
#print("Std:", segmented_vessels_array.std())

# Plot the raw vesselness image
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(segmented_vessels_array, cmap='gray')
plt.axis('off')
plt.title("Vesselness image")
plt.subplot(1, 2, 2)
plt.imshow(thresholded_vessels_ch2, cmap='gray')
plt.axis('off')
plt.title("Vessel mask")
plt.show()

# Show the results
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(input_image, cmap='gray')
plt.axis('off')
plt.title("Contrast image")
plt.subplot(1, 2, 2)
plt.imshow(input_image, cmap='gray')
plt.contour(thresholded_vessels_ch2, colors='red', linewidths=0.5, alpha=0.45)
plt.axis('off')
plt.title("Vessel mask contours over contrast image")
plt.show()

# Plot the raw vesselness image
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(segmented_vessels_array, cmap='gray')
plt.axis('off')
plt.title("Vesselness image")
plt.xlim(xlim)
plt.ylim(ylim)
plt.subplot(1, 2, 2)
plt.imshow(thresholded_vessels_ch2, cmap='gray')
plt.axis('off')
plt.title("Vessel mask")
plt.xlim(xlim)
plt.ylim(ylim)
plt.show()

# Show the results
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(input_image, cmap='gray')
plt.axis('off')
plt.title("Contrast image")
plt.xlim(xlim)
plt.ylim(ylim)
plt.subplot(1, 2, 2)
plt.imshow(input_image, cmap='gray')
plt.contour(thresholded_vessels_ch2, colors='red', linewidths=0.5, alpha=0.45)
plt.axis('off')
plt.title("Vessel mask contours over contrast image")
plt.xlim(xlim)
plt.ylim(ylim)
plt.show()

# Show the results
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(input_image, cmap='gray')
plt.axis('off')
plt.title("2: Contrast image")
plt.xlim(ylim)
plt.ylim(ylim)
plt.subplot(1, 2, 2)
plt.imshow(input_image, cmap='gray')
plt.contour(thresholded_vessels_ch2, colors='red', linewidths=0.5, alpha=0.45)
plt.axis('off')
plt.title("2: Vessel mask contours over contrast image")
plt.xlim(ylim)
plt.ylim(ylim)
plt.show()

# Show the results
plt.figure(figsize=(10, 10))
plt.subplot(1, 2, 1)
plt.imshow(input_image, cmap='gray')
plt.axis('off')
plt.title("3: Contrast image")
plt.xlim([2000, 3000])
plt.ylim([2000, 3000])
plt.subplot(1, 2, 2)
plt.imshow(input_image, cmap='gray')
plt.contour(thresholded_vessels_ch2, colors='red', linewidths=0.5, alpha=0.45)
plt.axis('off')
plt.title("3: Vessel mask contours over contrast image")
plt.xlim([2000, 3000])
plt.ylim([2000, 3000])
plt.show()

In [ ]:
# Compare side-by-side
plt.figure(figsize=(10, 10))
plt.subplot(1, 3, 1)
plt.imshow(thresholded_vessels_ch0, cmap='gray')
plt.axis('off')
plt.title("Vessel mask ch0")
plt.subplot(1, 3, 2)
plt.imshow(thresholded_vessels_ch1, cmap='gray')
plt.axis('off')
plt.title("Vessel mask ch1")
plt.subplot(1, 3, 3)
plt.imshow(thresholded_vessels_ch2, cmap='gray')
plt.axis('off')
plt.title("Vessel mask ch2")
plt.show()

np.save("mask_enhanced_ch0.npy", thresholded_vessels_ch0)
np.save("mask_enhanced_ch1.npy", thresholded_vessels_ch1)
np.save("mask_enhanced_ch2.npy", thresholded_vessels_ch2)

## Compute statistics 

In [5]:
import csv
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import mean_squared_error
from scipy.spatial.distance import hamming

In [ ]:
thresholded_ch0 = thresholded_vessels_ch0
thresholded_ch1 = thresholded_vessels_ch1
thresholded_ch2 = thresholded_vessels_ch2

# Compute the metrics
dice_score = dice_coefficient(thresholded_ch0, thresholded_ch1)  
iou_score = iou(thresholded_ch0, thresholded_ch1)  # Strongly penalizes over-segmentation and under-segmentation
precision_score = precision(thresholded_ch0, thresholded_ch1) 
recall_score = recall(thresholded_ch0, thresholded_ch1)
ssim_score = ssim(thresholded_ch0, thresholded_ch1)
mse_score = mean_squared_error(thresholded_ch0, thresholded_ch1)
thresh_ch0_flat = thresholded_ch0.flatten()
thresh_ch1_flat = thresholded_ch1.flatten()
hamming_distance = hamming(thresh_ch0_flat, thresh_ch1_flat)
rand_score = rand_index(thresholded_ch0, thresholded_ch1)  # Measures how close points are clustered together

print("Dice coefficient:", dice_score)
print("IoU score:", iou_score)
print("Precision score:", precision_score)
print("Recall score:", recall_score)
print("SSIM score:", ssim_score)
print("MSE score:", mse_score)
print("Hamming distance:", hamming_distance)
print("Rand index:", rand_score)

# Write the solutions to a CSV file
csv_filename = 'stats_enhanced.csv'

# Write to rows
rows = [["Index", "Dice coefficient", "IoU score", "Precision", "Recall", "SSIM", "MSE", "Hamming distance", "Rand index"]]
rows.append([IDX, dice_score, iou_score, precision_score, recall_score, ssim_score, mse_score, hamming_distance, rand_score])

with open(csv_filename, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(rows)

Run the whole thing

In [ ]:
from tqdm import tqdm

data_path = "/media/data/u01/Endothelial_enhancer/Fig3_M14_*.tif"
output_ch0_path = "/media/data/u01/lightsheet/quant2/M14/segmentation/ch0/"
output_ch1_path = "/media/data/u01/lightsheet/quant2/M14/segmentation/ch1/"
output_ch2_path = "/media/data/u01/lightsheet/quant2/M14/segmentation/ch2/"
output_csv_ch0_ch1_path = "/media/data/u01/lightsheet/quant2/M14/segmentation/stats_enhanced2_ch0_ch1.csv"
output_csv_ch0_ch2_path = "/media/data/u01/lightsheet/quant2/M14/segmentation/stats_enhanced2_ch0_ch2.csv"
output_csv_ch1_ch2_path = "/media/data/u01/lightsheet/quant2/M14/segmentation/stats_enhanced2_ch1_ch2.csv"

################################################################################

gamma_ch0 = 2  # You can adjust this value to control the contrast enhancement
contrast_alpha_ch0 = 0.5  # Try 0.15 You can adjust this value to control the brightness enhancement 0.5 default
gamma_ch1 = 2
contrast_alpha_ch1 = 1.5
gamma_ch2 = 2
contrast_alpha_ch2 = 1.5

# Parameters for vessel detection
sigma_minimum = 1.0  # Range of scales in which MultiScaleHessianBasedMeasureImageFilter will search for vessels
sigma_maximum = 10.0  # 10
number_of_sigma_steps = 10  # 10 Number of scales to search for vessels

# Parameters for post-processing
thresh = 230  # Threshold for binarization, 230
min_size = 500  # Minimum size of objects to keep
area_threshold = 2000 # Minimum area of holes to keep
smoothing = 1  # Smoothing factor for closing, 3

# Read all tif files in the folder
data_files = sorted(glob.glob(data_path))
rows_ch0_ch1 = [["Index", "Dice coefficient", "IoU score", "Precision", "Recall", "SSIM", "MSE", "Hamming distance", "Rand index"]]
rows_ch0_ch2 = [["Index", "Dice coefficient", "IoU score", "Precision", "Recall", "SSIM", "MSE", "Hamming distance", "Rand index"]]
rows_ch1_ch2 = [["Index", "Dice coefficient", "IoU score", "Precision", "Recall", "SSIM", "MSE", "Hamming distance", "Rand index"]]

for i in tqdm(range(len(data_files))):
    curr_img = read_tif(data_files[i])
    curr_ch0 = curr_img[0]
    curr_ch1 = curr_img[1]
    curr_ch2 = curr_img[2]
    
    # Compute the original image contrast
    contrast_ch0 = curr_ch0
    contrast_ch1 = curr_ch1
    contrast_ch2 = curr_ch2
    
    #contrast_ch0 = gamma_correction(curr_ch0, gamma=gamma_ch0)
    #contrast_ch0 = auto_contrast(contrast_ch0, alpha=contrast_alpha_ch0)
    #contrast_ch1 = gamma_correction(curr_ch1, gamma=gamma_ch1)
    #contrast_ch1 = auto_contrast(contrast_ch1, alpha=contrast_alpha_ch1)
    #contrast_ch2 = gamma_correction(curr_ch2, gamma=gamma_ch2)
    #contrast_ch2 = auto_contrast(contrast_ch2, alpha=contrast_alpha_ch2)
    
    bg_mask = gamma_correction(curr_ch0, gamma=gamma_ch0)
    bg_mask = auto_contrast(bg_mask, alpha=3.0)
    bg_mask = get_brain_mask(bg_mask, area_threshold=100000)
    
    #############################################################

    # Load image in memory
    input_ch0 = contrast_ch0 * bg_mask
    input_ch0 = input_ch0.astype(np.float32)
    #input_ch0 *= 255.0
    
    input_ch1 = contrast_ch1 * bg_mask
    input_ch1 = input_ch1.astype(np.float32)
    
    input_ch2 = contrast_ch2 * bg_mask
    input_ch2 = input_ch2.astype(np.float32)
    #input_ch1 *= 255.0

    # Run the vessel detection
    segmented_vessels_ch0 = detect_vessels(input_ch0, sigma_minimum, sigma_maximum, number_of_sigma_steps, alpha=0.5, beta=0.5, gamma=5.0)
    segmented_vessels_ch1 = detect_vessels(input_ch1, sigma_minimum, sigma_maximum, number_of_sigma_steps, alpha=0.5, beta=0.5, gamma=5.0)
    segmented_vessels_ch2 = detect_vessels(input_ch2, sigma_minimum, sigma_maximum, number_of_sigma_steps, alpha=0.5, beta=0.5, gamma=5.0)

    # Process the thresholded vessels
    thresholded_vessels_ch0 = process_vessels(segmented_vessels_ch0, thresh=thresh, min_size=min_size, area_threshold=area_threshold, smoothing=smoothing)
    thresholded_vessels_ch1 = process_vessels(segmented_vessels_ch1, thresh=thresh, min_size=min_size, area_threshold=area_threshold, smoothing=smoothing)
    thresholded_vessels_ch2 = process_vessels(segmented_vessels_ch2, thresh=thresh, min_size=min_size, area_threshold=area_threshold, smoothing=smoothing)

    thresholded_vessels_ch0 = thresholded_vessels_ch0 * bg_mask
    thresholded_vessels_ch1 = thresholded_vessels_ch1 * bg_mask
    thresholded_vessels_ch2 = thresholded_vessels_ch2 * bg_mask

    # Save to file
    sitk_ch0 = sitk.GetImageFromArray(thresholded_vessels_ch0.astype(np.uint8))  # Ch0
    output_ch0_file = output_ch0_path + f"ch0_seg_{str(i).zfill(4)}.tif"
    sitk.WriteImage(sitk_ch0, output_ch0_file)
    sitk_ch1 = sitk.GetImageFromArray(thresholded_vessels_ch1.astype(np.uint8))  # Ch1
    output_ch1_file = output_ch1_path + f"ch1_seg_{str(i).zfill(4)}.tif"
    sitk.WriteImage(sitk_ch1, output_ch1_file)
    sitk_ch2 = sitk.GetImageFromArray(thresholded_vessels_ch2.astype(np.uint8))  # Ch1
    output_ch2_file = output_ch2_path + f"ch2_seg_{str(i).zfill(4)}.tif"
    sitk.WriteImage(sitk_ch1, output_ch2_file)
    
    # Compute statistics between ch0 and ch1
    dice_score = dice_coefficient(thresholded_vessels_ch0, thresholded_vessels_ch1)  
    iou_score = iou(thresholded_vessels_ch0, thresholded_vessels_ch1)  # Strongly penalizes over-segmentation and under-segmentation
    precision_score = precision(thresholded_vessels_ch0, thresholded_vessels_ch1) 
    recall_score = recall(thresholded_vessels_ch0, thresholded_vessels_ch1)
    ssim_score = ssim(thresholded_vessels_ch0, thresholded_vessels_ch1)
    mse_score = mean_squared_error(thresholded_vessels_ch0, thresholded_vessels_ch1)
    thresh_ch0_flat = thresholded_vessels_ch0.flatten()
    thresh_ch1_flat = thresholded_vessels_ch1.flatten()
    hamming_distance = hamming(thresh_ch0_flat, thresh_ch1_flat)
    rand_score = rand_index(thresholded_vessels_ch0, thresholded_vessels_ch1)  # Measures how close points are clustered together
    rows_ch0_ch1.append([i, dice_score, iou_score, precision_score, recall_score, ssim_score, mse_score, hamming_distance, rand_score])
    print(rows_ch0_ch1[i + 1])
    
    # Compute statistics between ch0 and ch2
    dice_score = dice_coefficient(thresholded_vessels_ch0, thresholded_vessels_ch2)  
    iou_score = iou(thresholded_vessels_ch0, thresholded_vessels_ch2)  # Strongly penalizes over-segmentation and under-segmentation
    precision_score = precision(thresholded_vessels_ch0, thresholded_vessels_ch2) 
    recall_score = recall(thresholded_vessels_ch0, thresholded_vessels_ch2)
    ssim_score = ssim(thresholded_vessels_ch0, thresholded_vessels_ch2)
    mse_score = mean_squared_error(thresholded_vessels_ch0, thresholded_vessels_ch2)
    thresh_ch2_flat = thresholded_vessels_ch2.flatten()
    hamming_distance = hamming(thresh_ch0_flat, thresh_ch2_flat)
    rand_score = rand_index(thresholded_vessels_ch0, thresholded_vessels_ch2)  # Measures how close points are clustered together
    rows_ch0_ch2.append([i, dice_score, iou_score, precision_score, recall_score, ssim_score, mse_score, hamming_distance, rand_score])
    print(rows_ch0_ch2[i + 1])
    
    # Compute statistics between ch1 and ch2
    dice_score = dice_coefficient(thresholded_vessels_ch1, thresholded_vessels_ch2)  
    iou_score = iou(thresholded_vessels_ch1, thresholded_vessels_ch2)  # Strongly penalizes over-segmentation and under-segmentation
    precision_score = precision(thresholded_vessels_ch1, thresholded_vessels_ch2) 
    recall_score = recall(thresholded_vessels_ch1, thresholded_vessels_ch2)
    ssim_score = ssim(thresholded_vessels_ch1, thresholded_vessels_ch2)
    mse_score = mean_squared_error(thresholded_vessels_ch1, thresholded_vessels_ch2)
    hamming_distance = hamming(thresh_ch1_flat, thresh_ch2_flat)
    rand_score = rand_index(thresholded_vessels_ch1, thresholded_vessels_ch2)  # Measures how close points are clustered together
    rows_ch1_ch2.append([i, dice_score, iou_score, precision_score, recall_score, ssim_score, mse_score, hamming_distance, rand_score])
    print(rows_ch1_ch2[i + 1])
    
with open(output_csv_ch0_ch1_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(rows_ch0_ch1)
    
with open(output_csv_ch0_ch2_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(rows_ch0_ch2)
    
with open(output_csv_ch1_ch2_path, mode='w', newline='') as file:
    writer = csv.writer(file)
    writer.writerows(rows_ch1_ch2)